In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, explained_variance_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks
import joblib
import os
import optuna
from optuna.samplers import TPESampler
import time
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, RegressorMixin

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess the data
# Identify categorical and numerical columns
categorical_cols = ['current_stop_name', 'next_stop_name', 'day_of_week', 'weather_condition']
numerical_cols = ['is_holiday', 'is_peak_hour', 'passenger_count', 'current_speed', 
                  'distance_to_next_stop', 'current_lat', 'current_lon']

# Create preprocessors
numerical_scaler = StandardScaler()
# Updated parameter: sparse -> sparse_output
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
X_train_num = numerical_scaler.fit_transform(X_train[numerical_cols])
X_test_num = numerical_scaler.transform(X_test[numerical_cols])

X_train_cat = categorical_encoder.fit_transform(X_train[categorical_cols])
X_test_cat = categorical_encoder.transform(X_test[categorical_cols])

# Combine numerical and categorical features
X_train_processed = np.hstack((X_train_num, X_train_cat))
X_test_processed = np.hstack((X_test_num, X_test_cat))

# Reshape data for CNN (add channel dimension)
X_train_cnn = X_train_processed.reshape(X_train_processed.shape[0], X_train_processed.shape[1], 1)
X_test_cnn = X_test_processed.reshape(X_test_processed.shape[0], X_test_processed.shape[1], 1)

# Enhanced function to evaluate model with additional metrics
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    # Calculate MAPE with handling for zero or near-zero values
    # Filter out zero or very small values to avoid division by zero
    non_zero_indices = np.where(np.abs(y_true) > 0.1)  # Threshold of 0.1 minutes
    if len(non_zero_indices[0]) > 0:
        # Calculate MAPE only on non-zero values
        mape = np.mean(np.abs((y_true.iloc[non_zero_indices].values - y_pred[non_zero_indices]) / 
                             y_true.iloc[non_zero_indices].values)) * 100
    else:
        mape = np.nan  # or some placeholder value
    
    # Calculate alternative error metrics that don't suffer from division by zero
    # SMAPE (Symmetric Mean Absolute Percentage Error) is more robust to zero values
    smape = np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    
    # Calculate absolute errors for additional analysis
    abs_errors = np.abs(y_true - y_pred)
    max_error = np.max(abs_errors)
    median_error = np.median(abs_errors)
    
    # Calculate percentiles for error distribution
    p90_error = np.percentile(abs_errors, 90)
    p95_error = np.percentile(abs_errors, 95)
    p99_error = np.percentile(abs_errors, 99)
    
    # Count predictions within different error ranges
    within_1min = np.mean(abs_errors <= 1.0) * 100
    within_2min = np.mean(abs_errors <= 2.0) * 100
    within_5min = np.mean(abs_errors <= 5.0) * 100
    
    print(f"\n{model_name} Performance Metrics:")
    print(f"MAE: {mae:.4f} minutes")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f} minutes")
    print(f"MAPE (filtered): {mape:.2f}%")
    print(f"SMAPE: {smape:.2f}%")
    print(f"R² Score: {r2:.4f}")
    print(f"Explained Variance Score: {evs:.4f}")
    print(f"Median Error: {median_error:.4f} minutes")
    print(f"Maximum Error: {max_error:.4f} minutes")
    print(f"90th Percentile Error: {p90_error:.4f} minutes")
    print(f"95th Percentile Error: {p95_error:.4f} minutes")
    print(f"99th Percentile Error: {p99_error:.4f} minutes")
    print(f"Predictions within 1 minute: {within_1min:.2f}%")
    print(f"Predictions within 2 minutes: {within_2min:.2f}%")
    print(f"Predictions within 5 minutes: {within_5min:.2f}%")
    
    return {
        'MAE': mae, 
        'MSE': mse, 
        'RMSE': rmse, 
        'MAPE': mape,
        'SMAPE': smape,
        'R2': r2,
        'EVS': evs,
        'MedianError': median_error,
        'MaxError': max_error,
        'P90Error': p90_error,
        'P95Error': p95_error,
        'P99Error': p99_error,
        'Within1Min': within_1min,
        'Within2Min': within_2min,
        'Within5Min': within_5min
    }

# Define baseline CNN model
def create_baseline_cnn_model(input_shape):
    model = keras.Sequential([
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(50, activation='relu'),
        layers.Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train baseline CNN model
print("Training baseline CNN model...")
baseline_model = create_baseline_cnn_model((X_train_cnn.shape[1], 1))

# Use early stopping to prevent overfitting
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Track training time for baseline model
baseline_start_time = time.time()

baseline_history = baseline_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

baseline_training_time = time.time() - baseline_start_time
print(f"\nBaseline model training time: {baseline_training_time:.2f} seconds ({baseline_training_time/60:.2f} minutes)")

# Evaluate baseline model
baseline_prediction_start = time.time()
y_pred_baseline = baseline_model.predict(X_test_cnn).flatten()
baseline_prediction_time = time.time() - baseline_prediction_start
print(f"Baseline model prediction time (test set): {baseline_prediction_time:.4f} seconds")
print(f"Average prediction time per sample: {baseline_prediction_time/len(y_test)*1000:.4f} ms")

baseline_metrics = evaluate_model(y_test, y_pred_baseline, "Baseline Model")

# Optuna optimization for CNN
def objective(trial):
    # Define hyperparameters to optimize
    filters1 = trial.suggest_int('filters1', 16, 128)
    filters2 = trial.suggest_int('filters2', 16, 128)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dense_units = trial.suggest_int('dense_units', 16, 128)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    
    # Build model with the suggested hyperparameters
    model = keras.Sequential([
        layers.Conv1D(filters=filters1, kernel_size=kernel_size, activation='relu', 
                     input_shape=(X_train_cnn.shape[1], 1)),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        layers.Conv1D(filters=filters2, kernel_size=kernel_size, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        layers.Flatten(),
        layers.Dense(dense_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1)
    ])
    
    # Compile model
    optimizer = optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # Define callbacks
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    # Train model
    history = model.fit(
        X_train_cnn, y_train,
        epochs=100,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Get the validation loss from the epoch with the best performance
    val_loss = min(history.history['val_loss'])
    
    return val_loss

print("\nStarting Optuna optimization...")
optuna_start_time = time.time()
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)
optuna_time = time.time() - optuna_start_time
print(f"Optuna hyperparameter optimization completed in {optuna_time:.2f} seconds ({optuna_time/60:.2f} minutes)")

# Get the best hyperparameters
best_params = study.best_params
print("\nBest parameters:", best_params)

# Train optimized model with the best hyperparameters
print("\nTraining optimized model with best parameters...")
optimized_model = keras.Sequential([
    layers.Conv1D(filters=best_params['filters1'], kernel_size=best_params['kernel_size'], 
                 activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(best_params['dropout_rate']),
    layers.Conv1D(filters=best_params['filters2'], kernel_size=best_params['kernel_size'], 
                 activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(best_params['dropout_rate']),
    layers.Flatten(),
    layers.Dense(best_params['dense_units'], activation='relu'),
    layers.Dropout(best_params['dropout_rate']),
    layers.Dense(1)
])

# Compile model
optimizer = optimizers.Adam(learning_rate=best_params['learning_rate'])
optimized_model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Train model and track training time
optimized_start_time = time.time()

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

optimized_history = optimized_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

optimized_training_time = time.time() - optimized_start_time
print(f"\nOptimized model training time: {optimized_training_time:.2f} seconds ({optimized_training_time/60:.2f} minutes)")

# Evaluate optimized model prediction time
optimized_prediction_start = time.time()
y_pred_optimized = optimized_model.predict(X_test_cnn).flatten()
optimized_prediction_time = time.time() - optimized_prediction_start
print(f"Optimized model prediction time (test set): {optimized_prediction_time:.4f} seconds")
print(f"Average prediction time per sample: {optimized_prediction_time/len(y_test)*1000:.4f} ms")

optimized_metrics = evaluate_model(y_test, y_pred_optimized, "Optimized Model")

# Visualize learning curves
plt.figure(figsize=(12, 8))

# Plot training & validation loss
plt.subplot(2, 2, 1)
plt.plot(baseline_history.history['loss'], label='Baseline Training Loss')
plt.plot(baseline_history.history['val_loss'], label='Baseline Validation Loss')
plt.plot(optimized_history.history['loss'], label='Optimized Training Loss')
plt.plot(optimized_history.history['val_loss'], label='Optimized Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss (MSE)')
plt.xlabel('Epoch')
plt.legend()

# Plot training & validation MAE
plt.subplot(2, 2, 2)
plt.plot(baseline_history.history['mae'], label='Baseline Training MAE')
plt.plot(baseline_history.history['val_mae'], label='Baseline Validation MAE')
plt.plot(optimized_history.history['mae'], label='Optimized Training MAE')
plt.plot(optimized_history.history['val_mae'], label='Optimized Validation MAE')
plt.title('Model MAE')
plt.ylabel('MAE (minutes)')
plt.xlabel('Epoch')
plt.legend()

# Plot prediction error distribution
plt.subplot(2, 2, 3)
plt.hist(y_test - y_pred_baseline, bins=50, alpha=0.5, label='Baseline Model')
plt.hist(y_test - y_pred_optimized, bins=50, alpha=0.5, label='Optimized Model')
plt.title('Prediction Error Distribution')
plt.xlabel('Prediction Error (minutes)')
plt.ylabel('Frequency')
plt.legend()

# Plot actual vs predicted
plt.subplot(2, 2, 4)
plt.scatter(y_test, y_pred_baseline, alpha=0.3, label='Baseline Model')
plt.scatter(y_test, y_pred_optimized, alpha=0.3, label='Optimized Model')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'k--')
plt.title('Actual vs Predicted ETA')
plt.xlabel('Actual ETA (minutes)')
plt.ylabel('Predicted ETA (minutes)')
plt.legend()

plt.tight_layout()
plt.savefig('model_performance_comparison.png', dpi=300)
plt.close()

# Compare baseline and optimized models with expanded metrics
print("\nDetailed Performance Comparison:")
comparison_data = []
for metric_name in baseline_metrics.keys():
    baseline_val = baseline_metrics[metric_name]
    optimized_val = optimized_metrics[metric_name]
    
    if metric_name in ['MAE', 'MSE', 'RMSE', 'MAPE', 'MedianError', 'MaxError', 'P90Error', 'P95Error', 'P99Error']:
        # For these metrics, lower is better
        improvement = baseline_val - optimized_val
        improvement_pct = (improvement / baseline_val) * 100 if baseline_val != 0 else float('inf')
        better = improvement > 0
    else:
        # For R2, EVS, Within1Min, Within2Min, Within5Min, higher is better
        improvement = optimized_val - baseline_val
        improvement_pct = (improvement / baseline_val) * 100 if baseline_val != 0 else float('inf')
        better = improvement > 0
    
    comparison_data.append({
        'Metric': metric_name,
        'Baseline': baseline_val,
        'Optimized': optimized_val,
        'Improvement': improvement,
        'Improvement_Pct': improvement_pct,
        'Better': better
    })

# Convert to DataFrame and print
comparison_df = pd.DataFrame(comparison_data)
pd.set_option('display.float_format', '{:.4f}'.format)
print(comparison_df)

# Save comparison to CSV
comparison_df.to_csv('model_comparison_metrics.csv', index=False)
print("\nDetailed metrics comparison saved to model_comparison_metrics.csv")

# Visualize test predictions against actual values
plt.figure(figsize=(10, 6))
sample_size = min(500, len(y_test))  # Limit to 500 samples for clearer visualization
indices = np.random.choice(len(y_test), sample_size, replace=False)

plt.scatter(range(sample_size), y_test.iloc[indices], label='Actual', alpha=0.7, s=30)
plt.scatter(range(sample_size), y_pred_baseline[indices], label='Baseline Predictions', alpha=0.5, s=25)
plt.scatter(range(sample_size), y_pred_optimized[indices], label='Optimized Predictions', alpha=0.5, s=25)
plt.title('Test Set: Actual vs Predicted ETA Values')
plt.xlabel('Sample Index')
plt.ylabel('ETA (minutes)')
plt.legend()
plt.savefig('eta_predictions_comparison.png', dpi=300)
plt.close()

# Create a custom estimator wrapper for Keras model that follows scikit-learn API
class KerasModelWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, keras_model, preprocessor_num, preprocessor_cat, num_cols, cat_cols):
        self.keras_model = keras_model
        self.preprocessor_num = preprocessor_num
        self.preprocessor_cat = preprocessor_cat
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        
    def fit(self, X, y):
        # Already fitted, just a placeholder to comply with scikit-learn API
        return self
        
    def predict(self, X):
        # Apply preprocessing
        X_num = self.preprocessor_num.transform(X[self.num_cols])
        X_cat = self.preprocessor_cat.transform(X[self.cat_cols])
        X_processed = np.hstack((X_num, X_cat))
        X_reshaped = X_processed.reshape(X_processed.shape[0], X_processed.shape[1], 1)
        
        # Make predictions
        return self.keras_model.predict(X_reshaped).flatten()

# Create a wrapper for the optimized model
model_wrapper = KerasModelWrapper(
    optimized_model, 
    numerical_scaler, 
    categorical_encoder, 
    numerical_cols, 
    categorical_cols
)

# Calculate permutation feature importance using the wrapper
print("\nCalculating permutation feature importance...")
perm_importance_start = time.time()
r = permutation_importance(
    model_wrapper, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)
perm_importance_time = time.time() - perm_importance_start
print(f"Permutation importance calculated in {perm_importance_time:.2f} seconds")

# Create DataFrame of feature importances
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': r.importances_mean,
    'std': r.importances_std
}).sort_values('importance', ascending=False)

print("\nPermutation Feature Importance:")
print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Permutation Importance')
plt.title('Feature Importance for ETA Prediction')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.close()

# Save feature importance to CSV
feature_importance.to_csv('cnn_feature_importance.csv', index=False)
print("Feature importance saved to cnn_feature_importance.csv")

# Compare model sizes
baseline_params = baseline_model.count_params()
optimized_params = optimized_model.count_params()
print(f"\nModel Size Comparison:")
print(f"Baseline model parameters: {baseline_params:,}")
print(f"Optimized model parameters: {optimized_params:,}")
print(f"Difference: {optimized_params - baseline_params:,} parameters")

# Save the optimized model
model_filename = 'optimized_cnn_eta_predictor.keras'
optimized_model.save(model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Save an alternative format if needed
h5_model_filename = 'optimized_cnn_eta_predictor.h5'
optimized_model.save(h5_model_filename)
print(f"Optimized model also saved as {h5_model_filename}")

# Save the preprocessors for later use
preprocessor_filename = 'eta_preprocessors.pkl'
joblib.dump({
    'numerical_scaler': numerical_scaler,
    'categorical_encoder': categorical_encoder,
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols
}, preprocessor_filename)
print(f"Preprocessors saved as {preprocessor_filename}")

# Create a comprehensive performance report
performance_summary = {
    'Training Time (s)': [baseline_training_time, optimized_training_time],
    'Training Time (min)': [baseline_training_time/60, optimized_training_time/60],
    'Prediction Time (s)': [baseline_prediction_time, optimized_prediction_time],
    'Prediction Time per Sample (ms)': [baseline_prediction_time/len(y_test)*1000, optimized_prediction_time/len(y_test)*1000],
    'Model Size (parameters)': [baseline_params, optimized_params],
    'RMSE (minutes)': [baseline_metrics['RMSE'], optimized_metrics['RMSE']],
    'MAE (minutes)': [baseline_metrics['MAE'], optimized_metrics['MAE']],
    'SMAPE (%)': [baseline_metrics['SMAPE'], optimized_metrics['SMAPE']],
    'R²': [baseline_metrics['R2'], optimized_metrics['R2']],
    'Explained Variance': [baseline_metrics['EVS'], optimized_metrics['EVS']],
    'Within 1min (%)': [baseline_metrics['Within1Min'], optimized_metrics['Within1Min']],
    'Within 2min (%)': [baseline_metrics['Within2Min'], optimized_metrics['Within2Min']],
    'Within 5min (%)': [baseline_metrics['Within5Min'], optimized_metrics['Within5Min']]
}

summary_df = pd.DataFrame(performance_summary, index=['Baseline', 'Optimized'])
summary_df.loc['Improvement'] = summary_df.loc['Optimized'] - summary_df.loc['Baseline']
summary_df.loc['Improvement (%)'] = (summary_df.loc['Improvement'] / summary_df.loc['Baseline']) * 100

# Fix the sign for metrics where higher is better
for metric in ['R²', 'Explained Variance', 'Within 1min (%)', 'Within 2min (%)', 'Within 5min (%)']:
    if metric in summary_df.columns:
        summary_df.loc['Improvement (%)', metric] *= -1

print("\nComprehensive Performance Summary:")
print(summary_df)

# Save the summary
summary_df.to_csv('model_performance_summary.csv')
print("Performance summary saved to model_performance_summary.csv")

print("\nEvaluation complete! All performance metrics, visualizations, and models have been saved.")

C:\Users\cheng\AppData\Local\Temp\ipykernel_22312\740152611.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_22312\740152611.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_peak_hour'] = X['is_peak_hour'].astype(int)


Training baseline CNN model...
Epoch 1/100
2000/2000 [==============================] - 5s 2ms/step - loss: 1.0125 - mae: 0.5719 - val_loss: 0.3637 - val_mae: 0.3995
Epoch 2/100
2000/2000 [==============================] - 5s 2ms/step - loss: 0.4020 - mae: 0.4205 - val_loss: 0.3247 - val_mae: 0.3871
Epoch 3/100
2000/2000 [==============================] - 5s 2ms/step - loss: 0.3882 - mae: 0.4118 - val_loss: 0.3615 - val_mae: 0.3866
Epoch 4/100
2000/2000 [==============================] - 5s 2ms/step - loss: 0.3745 - mae: 0.4014 - val_loss: 0.4374 - val_mae: 0.4110
Epoch 5/100
2000/2000 [==============================] - 5s 2ms/step - loss: 0.3635 - mae: 0.3923 - val_loss: 0.3076 - val_mae: 0.3666
Epoch 6/100
2000/2000 [==============================] - 5s 2ms/step - loss: 0.3602 - mae: 0.3869 - val_loss: 0.3196 - val_mae: 0.3823
Epoch 7/100
2000/2000 [==============================] - 5s 2ms/step - loss: 0.3469 - mae: 0.3794 - val_loss: 0.3511 - val_mae: 0.3648
Epoch 8/100
2000/2000 [=

[I 2025-04-29 20:32:11,187] A new study created in memory with name: no-name-0717c2ee-5cc5-4d35-b964-1633a0856ca4


Baseline model prediction time (test set): 0.8669 seconds
Average prediction time per sample: 0.0433 ms

Baseline Model Performance Metrics:
MAE: 0.3454 minutes
MSE: 0.3093
RMSE: 0.5561 minutes
MAPE (filtered): 17.52%
SMAPE: 57.01%
R² Score: 0.9628
Explained Variance Score: 0.9628
Median Error: 0.2171 minutes
Maximum Error: 9.9123 minutes
90th Percentile Error: 0.7825 minutes
95th Percentile Error: 1.0610 minutes
99th Percentile Error: 2.0816 minutes
Predictions within 1 minute: 94.41%
Predictions within 2 minutes: 98.89%
Predictions within 5 minutes: 99.98%

Starting Optuna optimization...


[I 2025-04-29 20:34:35,613] Trial 0 finished with value: 0.3063962757587433 and parameters: {'filters1': 58, 'filters2': 123, 'kernel_size': 4, 'dense_units': 83, 'learning_rate': 0.0002051338263087451, 'batch_size': 64, 'dropout_rate': 0.3832290311184182}. Best is trial 0 with value: 0.3063962757587433.
[I 2025-04-29 20:35:36,251] Trial 1 finished with value: 0.5126994252204895 and parameters: {'filters1': 18, 'filters2': 125, 'kernel_size': 5, 'dense_units': 39, 'learning_rate': 0.0002310201887845295, 'batch_size': 64, 'dropout_rate': 0.21649165607921678}. Best is trial 0 with value: 0.3063962757587433.
[I 2025-04-29 20:40:48,223] Trial 2 finished with value: 0.29418444633483887 and parameters: {'filters1': 85, 'filters2': 31, 'kernel_size': 3, 'dense_units': 57, 'learning_rate': 0.000816845589476017, 'batch_size': 16, 'dropout_rate': 0.1185801650879991}. Best is trial 2 with value: 0.29418444633483887.
[I 2025-04-29 20:44:19,227] Trial 3 finished with value: 0.6477301716804504 and p

...

INFO:tensorflow:Assets written to: ram://6f7e6dae-77ce-4a46-9dcd-64a786435aa9/assets


INFO:tensorflow:Assets written to: ram://6f7e6dae-77ce-4a46-9dcd-64a786435aa9/assets


INFO:tensorflow:Assets written to: ram://432a7bcf-4ac4-45e2-889d-b682d904edb8/assets


INFO:tensorflow:Assets written to: ram://432a7bcf-4ac4-45e2-889d-b682d904edb8/assets


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.